In [37]:
!pip install yfinance scikit-learn pandas numpy -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR       = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')

os.makedirs(RAW_DIR,       exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f'RAW_DIR       : {RAW_DIR}')
print(f'PROCESSED_DIR : {PROCESSED_DIR}')


RAW_DIR       : c:\Users\akbar\VSCode Project\RaksaDana\data\raw
PROCESSED_DIR : c:\Users\akbar\VSCode Project\RaksaDana\data\processed


In [39]:
tickers    = ['BBCA.JK', 'BBRI.JK', 'BMRI.JK']
start_date = '2015-01-01'
end_date   = '2026-01-20'


In [40]:
raw = yf.download(tickers, start=start_date, end=end_date, progress=False)

price_data = {}
for ticker in tickers:
    df = raw.loc[:, (slice(None), ticker)].copy()
    df.columns = df.columns.droplevel(1)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df.index = pd.to_datetime(df.index)
    price_data[ticker] = df

In [41]:
price_data['BBCA.JK'].tail()

Price,Open,High,Low,Close,Volume
Date,,,,,
2026-01-12,7736.332185,7760.283678,7664.477706,7688.429199,82935700
2026-01-13,7688.429532,7736.332520,7664.478038,7736.332520,80536300
2026-01-14,7688.429521,7712.381015,7616.575040,7664.478027,105724300
2026-01-15,7664.478038,7832.138495,7640.526544,7736.332520,151753100
2026-01-19,7736.332365,7784.235352,7640.526391,7784.235352,172100000


In [42]:
# Save raw price data
for ticker in tickers:
    out_path = os.path.join(RAW_DIR, f'{ticker.replace(".", "_")}_raw.csv')
    price_data[ticker].to_csv(out_path)
    print(f"Saved: {out_path}")

Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\raw\BBCA_JK_raw.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\raw\BBRI_JK_raw.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\raw\BMRI_JK_raw.csv


In [43]:
fundamental = {}

for ticker in tickers:
    info = yf.Ticker(ticker).info
    fundamental[ticker] = {
        'ROE': info.get('returnOnEquity'),
        'EPS': info.get('trailingEps'),
        'DY' : info.get('dividendYield') / 100
    }

In [44]:
fundamental

{'BBCA.JK': {'ROE': 0.22972, 'EPS': 467.26, 'DY': 0.055099999999999996},
 'BBRI.JK': {'ROE': 0.18135999, 'EPS': 389.06, 'DY': 0.134},
 'BMRI.JK': {'ROE': 0.21040002, 'EPS': 603.12, 'DY': 0.11359999999999999}}

In [45]:
merged_data = {}

for ticker in tickers:
    df = price_data[ticker].copy()
    df['ROE'] = fundamental[ticker]['ROE']
    df['EPS'] = fundamental[ticker]['EPS']
    df['DY']  = fundamental[ticker]['DY']
    merged_data[ticker] = df

In [46]:
merged_data['BBCA.JK'].tail()

Price,Open,High,Low,Close,Volume,ROE,EPS,DY
Date,,,,,,,,
2026-01-12,7736.332185,7760.283678,7664.477706,7688.429199,82935700,0.22972,467.26,0.0551
2026-01-13,7688.429532,7736.332520,7664.478038,7736.332520,80536300,0.22972,467.26,0.0551
2026-01-14,7688.429521,7712.381015,7616.575040,7664.478027,105724300,0.22972,467.26,0.0551
2026-01-15,7664.478038,7832.138495,7640.526544,7736.332520,151753100,0.22972,467.26,0.0551
2026-01-19,7736.332365,7784.235352,7640.526391,7784.235352,172100000,0.22972,467.26,0.0551


In [47]:
def add_features(df):
    df = df.copy()
    
    df['MA7']  = df['Close'].rolling(7).mean()
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()

    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    sma20          = df['Close'].rolling(20).mean()
    std20          = df['Close'].rolling(20).std()
    df['BB_upper'] = sma20 + 2 * std20
    df['BB_lower'] = sma20 - 2 * std20
    df['BB_width'] = df['BB_upper'] - df['BB_lower']

    df['Daily_Return'] = df['Close'].pct_change()
    df['Log_Return']   = np.log(df['Close'] / df['Close'].shift(1))
    df['Volume_MA7']   = df['Volume'].rolling(7).mean()

    ema12          = df['Close'].ewm(span=12, adjust=False).mean()
    ema26          = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD']         = ema12 - ema26
    df['MACD_signal']  = df['MACD'].ewm(span=9, adjust=False).mean()

    df.dropna(inplace=True)
    return df

featured_data = {}
for ticker in tickers:
    featured_data[ticker] = add_features(merged_data[ticker])

In [48]:
featured_data['BBCA.JK'].columns.tolist()

['Open',
 'High',
 'Low',
 'Close',
 'Volume',
 'ROE',
 'EPS',
 'DY',
 'MA7',
 'MA20',
 'MA50',
 'RSI',
 'BB_upper',
 'BB_lower',
 'BB_width',
 'Daily_Return',
 'Log_Return',
 'Volume_MA7',
 'MACD',
 'MACD_signal']

In [49]:
feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'ROE', 'EPS', 'DY',
    'MA7', 'MA20', 'MA50',
    'RSI', 'BB_upper', 'BB_lower', 'BB_width',
    'Daily_Return', 'Log_Return', 'Volume_MA7',
    'MACD', 'MACD_signal',
]

WINDOW_SIZE = 60

scaled_train_data = {}
scaled_test_data  = {}
scalers           = {}

for ticker in tickers:
    df = featured_data[ticker][feature_cols].copy()

    n_total_seq   = len(df) - WINDOW_SIZE
    n_train_seq   = int(n_total_seq * 0.8)
    train_end_idx = WINDOW_SIZE + n_train_seq

    train_df = df.iloc[:train_end_idx]
    test_df  = df.iloc[train_end_idx - WINDOW_SIZE:]

    scaler = MinMaxScaler()
    scaler.fit(train_df)

    scaled_train_data[ticker] = pd.DataFrame(
        scaler.transform(train_df), columns=feature_cols, index=train_df.index,
    )
    scaled_test_data[ticker] = pd.DataFrame(
        scaler.transform(test_df), columns=feature_cols, index=test_df.index,
    )
    scalers[ticker] = scaler

    print(f'{ticker:10s}  train rows {len(train_df):4d}  test rows {len(test_df):4d}')


BBCA.JK     train rows 2150  test rows  583
BBRI.JK     train rows 2150  test rows  583
BMRI.JK     train rows 2150  test rows  583


In [50]:
scaled_train_data['BBCA.JK'].tail()


,Open,High,Low,Close,Volume,ROE,EPS,DY,MA7,MA20,MA50,RSI,BB_upper,BB_lower,BB_width,Daily_Return,Log_Return,Volume_MA7,MACD,MACD_signal
Date,,,,,,,,,,,,,,,,,,,,
2023-10-27,0.905702,0.911951,0.909265,0.905552,0.049349,0.0,0.0,0.0,0.926121,0.956362,0.980937,0.315556,0.974192,0.934801,0.224615,0.302101,0.328422,0.243996,0.524695,0.510286
2023-10-30,0.902334,0.918724,0.909265,0.925791,0.048885,0.0,0.0,0.0,0.928091,0.954791,0.979794,0.426824,0.972413,0.933461,0.223290,0.381741,0.410818,0.223728,0.532504,0.504735
2023-10-31,0.915805,0.915338,0.912625,0.912298,0.061069,0.0,0.0,0.0,0.923658,0.951649,0.978581,0.391984,0.967535,0.932124,0.213062,0.268694,0.293364,0.223918,0.529102,0.499447
2023-11-01,0.909070,0.908565,0.895823,0.892059,0.063063,0.0,0.0,0.0,0.918733,0.947459,0.976582,0.293412,0.963934,0.927390,0.216092,0.245550,0.268900,0.232767,0.511588,0.490857
2023-11-02,0.905702,0.925497,0.912626,0.925791,0.052736,0.0,0.0,0.0,0.920211,0.945889,0.975296,0.385576,0.960678,0.927552,0.206288,0.428591,0.458532,0.221479,0.526162,0.487613


In [51]:
TARGET_COL = 'Close'
TARGET_IDX = feature_cols.index(TARGET_COL)

def create_sequences(scaled_df, window_size, target_idx):
    X, y = [], []
    arr = scaled_df.values
    for i in range(window_size, len(arr)):
        X.append(arr[i - window_size : i, :])
        y.append(arr[i, target_idx])
    return np.array(X), np.array(y)

sequences = {}
for ticker in tickers:
    X_train, y_train = create_sequences(scaled_train_data[ticker], WINDOW_SIZE, TARGET_IDX)
    X_test,  y_test  = create_sequences(scaled_test_data[ticker],  WINDOW_SIZE, TARGET_IDX)
    sequences[ticker] = {
        'X_train': X_train, 'y_train': y_train,
        'X_test':  X_test,  'y_test':  y_test,
    }


In [52]:
for ticker in tickers:
    s = sequences[ticker]
    print(f"{ticker} → X_train: {s['X_train'].shape}, X_test: {s['X_test'].shape}")

BBCA.JK → X_train: (2090, 60, 20), X_test: (523, 60, 20)
BBRI.JK → X_train: (2090, 60, 20), X_test: (523, 60, 20)
BMRI.JK → X_train: (2090, 60, 20), X_test: (523, 60, 20)


In [53]:
output = {
    'sequences'    : sequences,
    'scalers'      : scalers,
    'feature_cols' : feature_cols,
    'target_col'   : TARGET_COL,
    'target_idx'   : TARGET_IDX,
    'window_size'  : WINDOW_SIZE,
    'featured_data': featured_data,
    'fundamental'  : fundamental,
    'tickers'      : tickers,
    'scaler_type'  : 'MinMaxScaler',
    'scaler_fit'   : 'train_only',
}

pkl_path = os.path.join(PROCESSED_DIR, 'preprocessed_data.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(output, f)
print(f'Saved: {pkl_path}')


Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\preprocessed_data.pkl


In [54]:
# Save featured CSVs
for ticker in tickers:
    out_path = os.path.join(PROCESSED_DIR, f'{ticker.replace(".", "_")}_featured.csv')
    featured_data[ticker].to_csv(out_path)
    print(f"Saved: {out_path}")

Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\BBCA_JK_featured.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\BBRI_JK_featured.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\BMRI_JK_featured.csv
